# Data Science Programming - Sesion 3: pandas

**Especializacion en Ciencia de Datos - Universidad Santo Tomas - Tunja**
**Prof. Hugo Hernandez | Martes 8 de septiembre, 2026**

---

## Que haremos hoy

1. **La tabla**: que es un DataFrame y que es una Series
2. **El indice**: la pieza que mas confunde, y por que no es la posicion
3. **Mirar antes de tocar**: head, info, describe
4. **Sacar columnas**: `df['col']`, `df[['a','b']]`, `.loc` y `.iloc`
5. **Sacar filas**: filtros booleanos con mascaras
6. **groupby**: separar en montoncitos y resumir cada montoncito
7. **Ordenar y contar**: sort_values y value_counts
8. **Datos reales**: el Titanic, con alternativa sin internet

> Ejecuta las celdas en orden con `Shift + Enter` o el boton de play.
> Experimenta: cambia valores y vuelve a ejecutar. Nada se rompe para siempre.

---
## Antes de empezar

- Ejecuta las celdas con **Shift + Enter**, de arriba hacia abajo.
- En **VS Code / Jupyter**, ejecuta la celda de abajo una vez para instalar las librerias.
- Cuando algo falle vas a ver un texto rojo (*traceback*). Leelo: ahi dice que paso.

In [ ]:
%pip install pandas seaborn --quiet
print("Librerias listas")

---
## Parte 1 - El DataFrame: una hoja de Excel que se programa

| En Excel | En pandas |
|---|---|
| Arrastras el mouse, filtras con menus | Escribes una linea de codigo |
| Manana repites todo el proceso a mano | Vuelves a ejecutar la misma celda |
| Se ahoga cerca del millon de filas | Aguanta millones sin despeinarse |
| Nadie sabe que hiciste para llegar al numero | El codigo **es** la memoria de lo que hiciste |

Un DataFrame se construye desde un diccionario: clave = nombre de columna,
valor = lista con el contenido de esa columna. Todas las listas miden lo mismo.

In [ ]:
import pandas as pd

datos = {
    'vendedor':  ['Ana', 'Luis', 'Sofia', 'Juan', 'Marta', 'Pedro',
                  'Elena', 'Carlos', 'Diana', 'Jorge', 'Laura', 'Ruben'],
    'ciudad':    ['Tunja', 'Bogota', 'Tunja', 'Cali', 'Bogota', 'Tunja',
                  'Duitama', 'Cali', 'Sogamoso', 'Duitama', 'Tunja', 'Bogota'],
    'categoria': ['Cafe', 'Cafe', 'Panela', 'Cafe', 'Panela', 'Arepa',
                  'Arepa', 'Panela', 'Queso', 'Cafe', 'Queso', 'Arepa'],
    'edad':      [23, 35, 29, 41, 38, 52, 27, 45, 33, 31, 26, 49],
    'unidades':  [120, 340, 90, 210, 180, 75, 150, 260, 60, 200, 45, 320],
    'precio':    [3500, 3500, 2800, 3500, 2800, 1500, 1500, 2800, 12000, 3500, 12000, 1500]
}

df = pd.DataFrame(datos)
df['total'] = df['unidades'] * df['precio']

df

### 1.1 La Series: una columna suelta

- **DataFrame** = la tabla completa.
- **Series** = una sola columna, con su etiqueta por cada valor.

Cuando algo falle, la primera pregunta es: esto es un DataFrame o una Series?

In [ ]:
print('df completo :', type(df))
print('una columna :', type(df['edad']))

print('\nForma de la tabla (filas, columnas):', df.shape)
print('Columnas:', list(df.columns))

print('\nTipo de cada columna:')
print(df.dtypes)

---
## Parte 2 - El indice

La columna sin titulo de la izquierda (0, 1, 2, 3...) no es una columna: es el **indice**,
el nombre propio de cada fila.

| | Indice | Posicion |
|---|---|---|
| Que es | La **etiqueta** de la fila | El **puesto** en que esta parada |
| Cambia si ordeno la tabla | No, viaja con su fila | Si, es el orden actual |
| Puede ser texto | Si | No, siempre entero |
| Como se accede | `.loc[...]` | `.iloc[...]` |

In [ ]:
print('Indice actual:', list(df.index))

ordenado = df.sort_values('total', ascending=False)
ordenado.head()

### 2.1 Poner nuestro propio indice

- `set_index('columna')` asciende esa columna a indice.
- `reset_index()` la devuelve a columna normal y repone 0, 1, 2, 3.

`set_index` no modifica el original: devuelve una tabla nueva. Hay que guardarla.

In [ ]:
por_vendedor = df.set_index('vendedor')

print('Indice nuevo:', list(por_vendedor.index))
print('Columnas que quedan:', list(por_vendedor.columns))

print('\nEl df original sigue con su indice:', list(df.index)[:5], '...')

por_vendedor.head()

---
## Parte 3 - El ritual de apertura

| Instruccion | Que responde |
|---|---|
| `df.head()` | Como se ven los datos? |
| `df.info()` | Cuantas filas? Que tipos? Faltan datos? |
| `df.describe()` | Que rango tienen los numeros? |

In [ ]:
df.head(3)

In [ ]:
print(df.tail(2))

print('\n--- info() ---')
df.info()

In [ ]:
df.describe()

---
## Parte 4 - Seleccionar columnas: un corchete o dos?

| Escribes | Recibes |
|---|---|
| `df['edad']` | Una **Series** |
| `df[['edad']]` | Un **DataFrame** de 1 columna |
| `df[['edad','ciudad']]` | Un **DataFrame** de 2 columnas |

Un corchete pide una cosa y devuelve una columna; dos corchetes piden una lista y
devuelven una tabla.

In [ ]:
una_columna = df['total']
print(type(una_columna))
print(una_columna.head(3))

print('\nVenta promedio: $', round(df['total'].mean()))
print('Venta maxima:   $', df['total'].max())
print('Suma total:     $', df['total'].sum())

In [ ]:
sub = df[['vendedor', 'ciudad', 'total']]
print(type(sub))
sub.head(4)

### 4.1 Y si el nombre de la columna no existe?

In [ ]:
print('Los nombres reales son:', list(df.columns))

df['ciudad_venta'].head(3)

---
## Parte 5 - .loc y .iloc

| | `.loc` | `.iloc` |
|---|---|---|
| Trabaja con | **Etiquetas** (el indice) | **Posiciones** (0, 1, 2...) |
| `[0]` significa | la fila **llamada** 0 | la **primera** fila |
| En un rango `[a:b]` | incluye el final | **no** incluye el final |

Forma completa: `[filas, columnas]`. Con `.loc` los dos son nombres; con `.iloc`, numeros.

In [ ]:
print('--- Una fila completa por su nombre ---')
print(por_vendedor.loc['Ana'])

print('\nLa ciudad de Ana es:', por_vendedor.loc['Ana', 'ciudad'])

print('\n--- Dos filas escogidas a dedo ---')
print(por_vendedor.loc[['Ana', 'Pedro'], ['ciudad', 'total']])

print('\n--- Rango de etiquetas: de Ana a Sofia ---')
print(por_vendedor.loc['Ana':'Sofia', ['ciudad', 'total']])

In [ ]:
print('--- La primera fila, sea quien sea ---')
print(por_vendedor.iloc[0])

print('\nFila 0, columna 1:', por_vendedor.iloc[0, 1])

print('\n--- Las tres primeras filas ---')
print(por_vendedor.iloc[0:3])

print('\nUltimo vendedor de la tabla:', por_vendedor.index[-1])

### 5.1 El choque: pedirle un numero a `.loc`

> **Nombre -> `.loc`. Numero de puesto -> `.iloc`.**

In [ ]:
print('Indice:', list(por_vendedor.index)[:4], '...')

print('\nPor POSICION, con iloc:')
print(por_vendedor.iloc[0])

por_vendedor.loc[0]

---
## Parte 6 - Filtrar filas: mascaras booleanas

Filtrar ocurre en dos tiempos:

```python
mascara = df['edad'] > 30      # tiempo 1: True/False por cada fila
df[mascara]                    # tiempo 2: solo las filas True
```

En la practica se escribe junto: `df[df['edad'] > 30]`.

In [ ]:
mascara = df['edad'] > 30
print('La mascara es una Series de booleanos:')
print(mascara.head(6))

print('\nCuantos cumplen la condicion:', mascara.sum())

mayores = df[mascara]
print('Filas que quedaron:', len(mayores))
mayores[['vendedor', 'edad', 'ciudad']]

In [ ]:
tunja = df[df['ciudad'] == 'Tunja']
print('Vendedores en Tunja:', len(tunja))

print(tunja[['vendedor', 'categoria', 'total']])

boyaca = df[df['ciudad'].isin(['Tunja', 'Duitama', 'Sogamoso'])]
print('\nVendedores en Boyaca:', len(boyaca))

### 6.1 Combinar condiciones: `&` y `|`, nunca `and` y `or`

| Quieres decir | En Python normal | En pandas |
|---|---|---|
| y (ambas) | `and` | `&` |
| o (alguna) | `or` | `\|` |
| no | `not` | `~` |

In [ ]:
adultos_tunja = df[(df['edad'] > 30) & (df['ciudad'] == 'Tunja')]
print('Mayores de 30 en Tunja:', len(adultos_tunja))
print(adultos_tunja[['vendedor', 'edad', 'ciudad']])

cafe_o_queso = df[(df['categoria'] == 'Cafe') | (df['categoria'] == 'Queso')]
print('\nVendedores de Cafe o Queso:', len(cafe_o_queso))

fuera_de_tunja = df[~(df['ciudad'] == 'Tunja')]
print('Vendedores fuera de Tunja:', len(fuera_de_tunja))

df[(df['edad'] > 30) and (df['ciudad'] == 'Tunja')]

### 6.2 Los parentesis no son adorno

`&` se resuelve **antes** que `>` y `==`. Sin parentesis, Python lee otra cosa
y lanza un `TypeError` que no menciona los parentesis por ningun lado.

Regla sin excepciones: **cada condicion, envuelta en su propio parentesis.**

```python
df[(df['edad'] > 30) & (df['ciudad'] == 'Tunja')]
```

In [ ]:
df[df['edad'] > 30 & df['ciudad'] == 'Tunja']

---
## Parte 7 - groupby: la operacion estrella

Separar en montoncitos y resumir cada montoncito: **split - apply - combine**.

```python
df.groupby('ciudad')['total'].sum()
```

| Pedazo | Que dice |
|---|---|
| `df` | de esta tabla |
| `.groupby('ciudad')` | haz montoncitos por ciudad |
| `['total']` | mirando la columna total |
| `.sum()` | y sumame cada montoncito |

In [ ]:
print('--- Venta total por ciudad ---')
print(df.groupby('ciudad')['total'].sum())

print('\n--- Venta PROMEDIO por ciudad ---')
print(df.groupby('ciudad')['total'].mean().round(0))

print('\n--- Cuantos vendedores por ciudad ---')
print(df.groupby('ciudad').size())

In [ ]:
print('--- Venta por ciudad y categoria ---')
print(df.groupby(['ciudad', 'categoria'])['total'].sum())

print('\n--- Varias cuentas al tiempo, por categoria ---')
print(df.groupby('categoria')['total'].agg(['count', 'sum', 'mean', 'max']).round(0))

### 7.1 El descuido clasico: agrupar y promediar *todo*

- Bien: `df.groupby('ciudad')['total'].mean()` -> dices que columna resumir.
- Atajo: `df.groupby('ciudad').mean(numeric_only=True)` -> solo para explorar.

In [ ]:
df.groupby('ciudad').mean()

---
## Parte 8 - Ordenar y contar

- `sort_values('columna')` ordena. Con `ascending=False`, de mayor a menor.
- `value_counts()` cuenta cuantas veces aparece cada valor de una columna.

In [ ]:
print('--- Top 5 en ventas ---')
print(df.sort_values('total', ascending=False)[['vendedor', 'ciudad', 'total']].head(5))

print('\n--- Por ciudad y, dentro de ella, por venta ---')
print(df.sort_values(['ciudad', 'total'], ascending=[True, False])[['ciudad', 'vendedor', 'total']])

In [ ]:
print('--- Vendedores por ciudad ---')
print(df['ciudad'].value_counts())

print('\n--- La misma cuenta, en proporcion ---')
print(df['categoria'].value_counts(normalize=True).round(3))

---
## Parte 9 - Datos reales: el Titanic

891 pasajeros: edad, clase, tarifa y si sobrevivieron.

La celda intenta descargarlo con `seaborn`. Si no hay internet, el `try / except` arma una
muestra de 12 pasajeros hecha a mano y la clase sigue igual.

In [ ]:
try:
    import seaborn as sns
    titanic = sns.load_dataset('titanic')
    print('Titanic cargado desde seaborn. Filas y columnas:', titanic.shape)
except Exception:
    print('Sin internet. Usamos una muestra de juguete de 12 pasajeros.')
    titanic = pd.DataFrame({
        'survived': [0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0],
        'pclass':   [3, 1, 3, 1, 3, 3, 1, 3, 2, 2, 3, 1],
        'sex':      ['male', 'female', 'female', 'female', 'male', 'male',
                     'male', 'female', 'female', 'female', 'male', 'male'],
        'age':      [22.0, 38.0, 26.0, 35.0, 35.0, 54.0, 2.0, 27.0, 14.0, 4.0, 20.0, 58.0],
        'fare':     [7.25, 71.28, 7.92, 53.10, 8.05, 51.86, 21.07, 11.13, 30.07, 16.70, 8.05, 26.55]
    })

titanic.head()

In [ ]:
titanic.info()

In [ ]:
print('--- Supervivencia por clase ---')
print(titanic.groupby('pclass')['survived'].mean().round(3))

print('\n--- Supervivencia por sexo ---')
print(titanic.groupby('sex')['survived'].mean().round(3))

print('\n--- Por sexo y clase a la vez ---')
print(titanic.groupby(['sex', 'pclass'])['survived'].mean().round(3))

print('\n--- Edad promedio de los pasajeros ---')
print(round(titanic['age'].mean(), 1), 'anios')

### 9.1 Leer el resultado, que es lo que de verdad importa

En primera clase sobrevivio cerca del 63%; en tercera, cerca del 24%. Y las mujeres
mucho mas que los hombres, en las tres clases.

Calcular es lo facil. Escoger la pregunta y leer la respuesta es el oficio.
Cada tabla que entregues deberia venir con una frase tuya diciendo que significa.

---
## Parte 10 - DEMO: juntamos todo

Seleccion + filtro + groupby + orden + value_counts, en el orden en que se hace de verdad.

Pregunta: que ciudad de Boyaca conviene reforzar el proximo trimestre?

In [ ]:
CIUDADES_BOYACA = ['Tunja', 'Duitama', 'Sogamoso']
boyaca = df[df['ciudad'].isin(CIUDADES_BOYACA)]
print('Filas en Boyaca:', len(boyaca), 'de', len(df), 'en total')

resumen = boyaca.groupby('ciudad')['total'].agg(['count', 'sum', 'mean']).round(0)
resumen.columns = ['vendedores', 'venta_total', 'venta_promedio']

resumen = resumen.sort_values('venta_total', ascending=False)
print('\n--- Resumen por ciudad ---')
print(resumen)

lider = resumen.index[0]
print(f'\nLa ciudad con mas venta acumulada es {lider}.')

print('\n--- Que se vende en Boyaca ---')
print(boyaca['categoria'].value_counts())

---
---
## TAREA - antes del martes 15 de septiembre

### Parte 1 - Trabaja sobre la tabla de notas

In [ ]:
import pandas as pd

notas = pd.DataFrame({
    'estudiante': ['Ana', 'Luis', 'Sofia', 'Juan', 'Marta', 'Pedro',
                   'Elena', 'Carlos', 'Diana', 'Jorge'],
    'programa':   ['Datos', 'Datos', 'Sistemas', 'Sistemas', 'Datos', 'Industrial',
                   'Industrial', 'Sistemas', 'Datos', 'Industrial'],
    'semestre':   [1, 3, 2, 4, 1, 3, 2, 4, 3, 1],
    'nota':       [4.2, 3.1, 4.8, 2.9, 3.7, 4.5, 3.3, 4.0, 2.6, 3.9],
    'asistencia': [95, 70, 98, 60, 88, 92, 75, 85, 55, 90]
})

# TODO 1: aplica el ritual de apertura: head(), info() y describe().
#
# TODO 2: muestra solo las columnas 'estudiante' y 'nota' (recuerda: doble corchete).
#
# TODO 3: pon 'estudiante' como indice y saca la fila de Sofia con .loc
#
# TODO 4: filtra los estudiantes que aprobaron (nota mayor o igual a 3.0).
#
# TODO 5: filtra los que aprobaron Y ademas tienen asistencia mayor a 80.
#         Cada condicion en su parentesis, unidas con &.
#
# TODO 6: nota promedio por programa, con groupby.
#
# TODO 7: ordena la tabla por nota de mayor a menor y muestra las 3 primeras.
#
# TODO 8: cuenta cuantos estudiantes hay por programa con value_counts().
#
# TODO 9: escribe en un comentario que programa va mejor y por que lo dices.

notas.head()

### Parte 2 - Responde

1. Cual es la diferencia entre un **DataFrame** y una **Series**? Como lo compruebas en codigo?
2. Que devuelve `df['edad']` y que devuelve `df[['edad']]`? Por que no son lo mismo?
3. Por que `.loc[0]` puede fallar y `.iloc[0]` no? Que significa exactamente `KeyError: 0`?
4. Por que hay que usar `&` en vez de `and` al combinar dos condiciones? Copia el mensaje de
   error que sale si usas `and`.
5. Explica `groupby` con un ejemplo de tu propia vida o de tu trabajo, sin usar la palabra
   pandas.

**Tus respuestas:**

1.
2.
3.
4.
5.

---
## Resumen

| Concepto | Ejemplo | Para que |
|---|---|---|
| Crear tabla | `pd.DataFrame(diccionario)` | armar datos a mano |
| Ver forma | `df.shape` | cuantas filas y columnas hay |
| Ritual de apertura | `.head() .info() .describe()` | conocer antes de calcular |
| Una columna (Series) | `df['edad']` | trabajar con una variable |
| Varias columnas | `df[['edad','ciudad']]` | recortar la tabla |
| Fila por etiqueta | `df.loc['Ana']` | acceder por nombre |
| Fila por posicion | `df.iloc[0]` | acceder por puesto |
| Cambiar el indice | `df.set_index('col')` | usar una columna como nombre de fila |
| Mascara | `df['edad'] > 30` | Series de True/False |
| Filtrar | `df[df['edad'] > 30]` | quedarse con unas filas |
| Dos condiciones | `df[(cond1) & (cond2)]` | combinar filtros |
| Esta en la lista | `df['col'].isin([...])` | filtrar por varios valores |
| Agrupar y resumir | `df.groupby('c')['t'].sum()` | responder por categoria |
| Varias cuentas | `.agg(['count','mean'])` | resumen completo de un golpe |
| Ordenar | `df.sort_values('col', ascending=False)` | ranking |
| Contar valores | `df['col'].value_counts()` | conocer una columna de texto |

### Los cinco errores que viste hoy

| Error | Que significa |
|---|---|
| `KeyError: 'nombre'` | pediste una columna que no existe (revisa `df.columns`) |
| `KeyError: 0` | usaste `.loc` con un numero sobre un indice de texto |
| `ValueError: truth value ... ambiguous` | usaste `and` u `or` donde van `&` y `\|` |
| `TypeError: unsupported operand ... &` | te faltaron los parentesis alrededor de cada condicion |
| `TypeError: ... does not support operation 'mean'` | promediaste una columna de texto |

### Las tres reglas para no sufrir con pandas

1. **Mira antes de tocar**: `head`, `info`, `describe`, y lee `df.columns` en vez de escribir de memoria.
2. **Nombre -> `.loc`, puesto -> `.iloc`.** El indice no es la posicion.
3. **Cada condicion en su parentesis, unidas con `&` o `|`.** Nunca `and` ni `or`.

**Proxima sesion:** limpieza y transformacion - datos faltantes, duplicados, outliers y
escalado. Es el corazon de la Actividad 2.

*Dudas: consulta al profesor*

---
---
## Ejercicios de refuerzo (opcionales)

Practica por tu cuenta sobre el DataFrame `df` de los vendedores. Van de mas facil a mas
retador. Intenta primero sin mirar la pista.

### Ejercicio 1 - Facil
Muestra las columnas `vendedor`, `categoria` y `unidades`, ordenadas por `unidades` de mayor
a menor. Cuantas unidades vendio el primero?

> *Pista: doble corchete para varias columnas, y `sort_values('unidades', ascending=False)`.*

In [ ]:
# Tu solucion aqui


### Ejercicio 2 - Medio
Cuanto vendio en total cada categoria, y cual fue la mas fuerte? Muestra el resultado
ordenado de mayor a menor.

> *Pista: `groupby('categoria')['total'].sum()` y despues `.sort_values(ascending=False)`.*

In [ ]:
# Tu solucion aqui


### Ejercicio 3 - Reta
Escribe una funcion `resumen_ciudad(tabla, nombre_ciudad)` que reciba el DataFrame y el
nombre de una ciudad, y devuelva un diccionario con tres claves: `vendedores` (cuantos hay),
`venta_total` y `venta_promedio`. Pruebala con 'Tunja' y con 'Bogota'.

> *Pista: filtra primero con una mascara, y de ahi saca `len()`, `.sum()` y `.mean()`.
> Recuerda el `return` de la Sesion 1: sin el, la funcion devuelve `None`.*

In [ ]:
# Tu solucion aqui
